w# Reddit Sentiment Analysis Project

Collecting comments from r/ChatGPT, r/ClaudeAI, and r/Gemini to analyze public sentiment towards AI models.



Importing Libraries

In [74]:
#import libraries
import pandas as pd

In [81]:
#read the file

rdf = pd.read_csv('/Users/muharremokutan/reddit_data_raw 1.csv')

/var/folders/c6/0mll9cbn1qvf3wt7cs7bpfl40000gn/T/ipykernel_56124/2681190434.py:3: DtypeWarning: Columns (0: post_title) have mixed types. Specify dtype option on import or set low_memory=False.
  rdf = pd.read_csv('/Users/muharremokutan/reddit_data_raw 1.csv')


In [82]:
# Check the data
print(rdf.shape)
print(rdf.head())

(1403180, 8)
  subreddit  post_id post_title     type  \
0   ChatGPT  1ncz57y        NaN  comment   
1   ChatGPT  1ne13tx        NaN  comment   
2   ChatGPT  1nebx17        NaN  comment   
3   ChatGPT  1ndtzc8        NaN  comment   
4   ChatGPT  1nejs7v        NaN  comment   

                                                text  score  upvote_ratio  \
0  https://preview.redd.it/1lh448otdmof1.png?widt...      2           NaN   
1  You replace one neuron with a machine like cou...      1           NaN   
2  that's an eye opening (not open a eyeing) pers...      2           NaN   
3  https://preview.redd.it/tlr7i3b0emof1.png?widt...      3           NaN   
4  The bot should be public domain with public co...      2           NaN   

         date  
0  2025-09-11  
1  2025-09-11  
2  2025-09-11  
3  2025-09-11  
4  2025-09-11  


Step 1: Check the baseline

In [87]:
print(f"Total rows: {len(rdf)}")
print(f"Total columns: {len(rdf.columns)}")
print(f"\nColumn names: {rdf.columns.tolist()}")

Total rows: 1345392
Total columns: 8

Column names: ['subreddit', 'post_id', 'post_title', 'type', 'text', 'score', 'upvote_ratio', 'date']


Task 1: Deduplicate Rows

In [107]:
# Count duplicates (same post_id + text)
duplicates = rdf.duplicated(subset=['post_id', 'text'], keep='first')
print(f"Duplicate rows to remove: {duplicates.sum():,}")


Duplicate rows to remove: 0


Task 2: Drop [removed] and [deleted]

In [109]:
# Check for exact matches
removed_exact = (rdf['text'] == '[removed]').sum()
deleted_exact = (rdf['text'] == '[deleted]').sum()

print(f"'[removed]': {removed_exact:,}")
print(f"'[deleted]': {deleted_exact:,}")

# Check for variations (different cases, spaces)
text_stripped = rdf['text'].fillna('').str.strip().str.lower()
removed_variations = text_stripped.isin(['[removed]', 'removed', '[removed by reddit]']).sum()
deleted_variations = text_stripped.isin(['[deleted]', 'deleted', '[deleted by user]']).sum()

print(f"\nWith variations '[removed]': {removed_variations:,}")
print(f"With variations '[deleted]': {deleted_variations:,}")


'[removed]': 0
'[deleted]': 0

With variations '[removed]': 1
With variations '[deleted]': 9


 Unique values that might be removed/deleted

In [108]:
print("\nUnique short texts:")
short_texts = rdf[rdf['text'].str.len() < 20]['text'].value_counts().head(20)
print(short_texts)


Unique short texts:
text
Yes           561
😂             494
Same          453
lol           409
Lol           350
No            342
🤣             287
Thank you!    272
Lmao          248
Thanks!       241
😂😂😂           199
Thanks        199
Yes.          186
🤣🤣🤣           186
Why?          176
No.           171
Thank you     167
What?         162
same          158
LOL           155
Name: count, dtype: int64


Task 3: Handle Nulls/Empty Text

In [110]:
# Count nulls
null_count = rdf['text'].isna().sum()
print(f"Null text fields: {null_count:,}")

Null text fields: 0


In [111]:

# Count empty strings
empty_count = (rdf['text'] == '').sum()
print(f"Empty string text fields: {empty_count:,}")

Empty string text fields: 0


In [112]:

# Count whitespace-only
whitespace_count = (rdf['text'].fillna('').str.strip() == '').sum()
print(f"Whitespace-only text fields: {whitespace_count:,}")

Whitespace-only text fields: 0


Data is already very clean! Only 10 rows have "removed" or "deleted" variations. We are removing them.

In [ ]:
print(f"Rows before cleaning: {len(rdf):,}")

In [106]:
# Task 1: Drop removed/deleted variations
text_stripped = rdf_clean['text'].fillna('').str.strip().str.lower()
remove_mask = text_stripped.isin(['[removed]', 'removed', '[removed by reddit]',
                                   '[deleted]', 'deleted', '[deleted by user]'])
rdf_clean = rdf_clean[~remove_mask]
print(f"After removing [removed]/[deleted]: {len(rdf_clean):,}")

# Summary
print(f"\n{'='*40}")
print(f"Total rows removed: {len(rdf) - len(rdf_clean):,}")
print(f"Final clean dataset: {len(rdf_clean):,} rows")

Rows before cleaning: 1,345,392
After removing [removed]/[deleted]: 1,345,382

Total rows removed: 10
Final clean dataset: 1,345,382 rows


In [114]:
# Save cleaned data
rdf_clean.to_csv('/Users/muharremokutan/reddit_data_cleaned.csv', index=False)

Saved 1,345,382 rows to reddit_data_cleaned.csv
